In [6]:

from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
import getpass
import os

import dotenv
dotenv.load_dotenv()

DB_URL = "sqlite:///./database.sqlite"

db = SQLDatabase.from_uri(
    DB_URL,
)



if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")


toolkit = SQLDatabaseToolkit(db=db, llm=llm)

toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000017D866C4A60>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000017D866C4A60>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000017D866C4A60>),
 QuerySQLCheckerTool(description='Use this tool to 

In [7]:
from langchain import hub

prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")

assert len(prompt_template.messages) == 1
print(prompt_template.input_variables)

['dialect', 'top_k']


In [8]:
system_message = prompt_template.format(dialect="SQLite", top_k=5)

In [ ]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(llm, toolkit.get_tools(), prompt=system_message)


In [10]:
example_query = "What are the names of all the tables in the database?"

events = agent_executor.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What are the names of all the tables in the database?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_N7EUF5LCbmk4a5eMYvHnpioL)
 Call ID: call_N7EUF5LCbmk4a5eMYvHnpioL
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Ball_by_Ball, Batsman_Scored, Batting_Style, Bowling_Style, City, Country, Extra_Runs, Extra_Type, Match, Out_Type, Outcome, Player, Player_Match, Rolee, Season, Team, Toss_Decision, Umpire, Venue, Wicket_Taken, Win_By, sysdiagrams
================================== Ai Message ==================================

The tables in the database are:

1. Ball_by_Ball
2. Batsman_Scored
3. Batting_Style
4. Bowling_Style
5. City
6. Country
7. Extra_Runs
8. Extra_Type
9. Match
10. Out_Type
11. Outcome
12. Player
13. Player_Match
14. Rolee
15. Season
16. Team
17. Toss_